In [1]:
import os
import numpy as np
import scipy.sparse as sp


from quant_rotor.core.dense.t_amplitudes_periodic import t_periodic
from quant_rotor.core.dense.t_amplitudes_periodic_fast import t_periodic as t_periodic_fast
from quant_rotor.core.dense.t_amplitudes_guess import t_1_amplitude_guess_ground_state, t_2_amplitude_guess_ground_state

from quant_rotor.core.dense.t_amplitudes_periodic_2D import t_periodic as t_periodic_2d

# Dense (Slow)
from quant_rotor.core.dense.hamiltonian import hamiltonian_dense
from quant_rotor.core.dense.hamiltonian_big import hamiltonian_general_dense

# Sparse (Fast)
from quant_rotor.core.sparse.hamiltonian import hamiltonian_sparse
from quant_rotor.core.sparse.hamiltonian_big import hamiltonian_general_sparse

from quant_rotor.models.dense.support_ham import (
    H_kinetic,
    H_potential_double,
    V_double,
    basis_m_to_p_matrix_conversion,
    write_matrix_elements,
    interaction_general_angle
)

from quant_rotor.models.sparse.support_ham import (build_V_in_p, H_potential_sparse)

In [2]:
np.set_printoptions(precision=5)
np.set_printoptions(suppress=True)
np.set_printoptions(linewidth=np.inf)
np.set_printoptions(threshold=np.inf)

In [3]:
def one_electron(i1: int, i2: int, j1: int, j2: int) -> complex:

    if (abs(i1 - j1) != 1) or (abs(i2 - j2) != 1):
        return 0.0


    if i1 == j1 + 1:
        if i2 == j2 + 1:

            return 0
        else:

            return 1
    else:
        if i2 == j2 + 1:
            return 1
        else:
            return 0

In [4]:
def write_matrix_elements_new(
    d: int, psi_twist: float = 0
) -> tuple[np.ndarray, np.ndarray]:
    """
    Construct kinetic and potential energy operator matrices for a truncated rotor basis.

    Parameters
    ----------
    numer_unique_states : int
        Number of unique momentum states to include in the truncated basis.
        The total dimension of the basis is given by:
        d = 2 * numer_unique_states + 1.
    tau : float
        Dipolar plains chain angle.

    Returns
    -------
    tuple[np.ndarray, np.ndarray]
        - K: diagonal kinetic energy operator as a dense CSR matrix in the 'p' basis
        - V: dense potential energy operator as a CSR matrix in the 'm' basis
    """

    # Generate Potential Energy Matrix
    V = np.zeros((d**2, d**2), dtype=complex)
    for i in range(d):
        for j in range(d):
            for k in range(d):
                for l in range(d):
                    # if k * d + l >= i * d + j:
                    V[i * d + j, k * d + l] = one_electron(
                        i, j, k, l
                    )

    return V

# Test 2D iterative CCC 

In [17]:
site = 3

state = 5
g = 0.1

In [18]:
t_1_max_2d, t_2_max_2d, energy_2d, t_1_2d, t_2_2d = t_periodic_2d(site, state, g, 0, 1)

In [19]:
t_1_max, t_2_max, energy, t_1, t_2 = t_periodic(site, state, g, True)

0j
0j
0j
0j
0j
0j
0j
0j
0j


In [20]:
H, K, V = hamiltonian_dense(state, site, g)

eig_val, eig_vec = np.linalg.eigh(H)

In [21]:
energy_2d

np.complex128(-0.012943004386135295+0j)

In [22]:
energy*3

np.complex128(-0.060605540670137584+0j)

In [23]:
np.min(eig_val)

np.float64(-0.020101689104206854)

## Combined interactions

In [14]:
dim_x = 3

state = 3
g = 0.1

In [15]:
t_1_max_2d_h, t_2_max_2d_h, energy_2d_h, t_1_2d_h, t_2_2d_h = t_periodic_2d(dim_x, state, g, 1, 0)
t_1_max_2d_v, t_2_max_2d_v, energy_2d_v, t_1_2d_v, t_2_2d_v = t_periodic_2d(dim_x, state, g, 0, 1)
t_1_max_2d_d, t_2_max_2d_d, energy_2d_d, t_1_2d_d, t_2_2d_d = t_periodic_2d(dim_x, state, g, 1, 1)

In [16]:
A_nz = t_2_2d_h != 0      # True where A is non-zero
B_nz = t_2_2d_v != 0
C_nz = t_2_2d_d != 0

In [17]:
t_2_sum = t_2_2d_h + t_2_2d_v + t_2_2d_d

In [18]:
for i in range(dim_x):
    print(np.allclose(t_2_sum[0], t_2_sum[i], atol=1e-1))

True
False
False


In [19]:
t_2_2d_h[0]

array([[[[[0.     +0.j]],

         [[0.     +0.j]]],


        [[[0.     +0.j]],

         [[0.     +0.j]]]],



       [[[[0.03954+0.j]],

         [[0.01581+0.j]]],


        [[[0.01581+0.j]],

         [[0.03954+0.j]]]],



       [[[[0.03954+0.j]],

         [[0.01581+0.j]]],


        [[[0.01581+0.j]],

         [[0.03954+0.j]]]],



       [[[[0.00217+0.j]],

         [[0.00336+0.j]]],


        [[[0.00336+0.j]],

         [[0.00217+0.j]]]],



       [[[[0.00109+0.j]],

         [[0.00168+0.j]]],


        [[[0.00168+0.j]],

         [[0.00109+0.j]]]],



       [[[[0.     +0.j]],

         [[0.     +0.j]]],


        [[[0.     +0.j]],

         [[0.     +0.j]]]],



       [[[[0.     +0.j]],

         [[0.     +0.j]]],


        [[[0.     +0.j]],

         [[0.     +0.j]]]],



       [[[[0.     +0.j]],

         [[0.     +0.j]]],


        [[[0.     +0.j]],

         [[0.     +0.j]]]],



       [[[[0.     +0.j]],

         [[0.     +0.j]]],


        [[[0.     +0.j]],

     

In [20]:
t_2_2d_v[0].shape

(9, 2, 2, 1, 1)

In [21]:
all_nonzero = A_nz & B_nz & C_nz
all_zero = (~A_nz) & (~B_nz) & (~C_nz)

A_n_B_z_C_z = A_nz & (~B_nz) & (~C_nz)
A_z_B_n_C_z = (~A_nz) & B_nz & (~C_nz)
A_z_B_z_C_n = (~A_nz) & (~B_nz) & C_nz

A_n_B_n_C_z = A_nz & B_nz & (~C_nz)
A_z_B_n_C_n = (~A_nz) & B_nz & C_nz
A_n_B_z_C_n = A_nz & (~B_nz) & C_nz

In [22]:
print(np.sum(all_zero))
print(np.sum(all_nonzero))

print(np.sum(A_n_B_z_C_z))
print(np.sum(A_z_B_n_C_z))
print(np.sum(A_z_B_z_C_n))

print(np.sum(A_n_B_n_C_z))
print(np.sum(A_z_B_n_C_n))
print(np.sum(A_n_B_z_C_n))

92
32
56
56
56
0
32
0


In [23]:
np.sum(A_z_B_z_C_n) + np.sum(A_z_B_n_C_z) + np.sum(A_n_B_z_C_z) + np.sum(all_zero) + 64

np.int64(324)

# Test 3D iterative CCC 

In [24]:
dim_x = 3

state = 3
g = 0.1

In [25]:
t_1_max_2d, t_2_max_2d, energy_2d, t_1_2d, t_2_2d = t_periodic_2d(dim_x, state, g)

TypeError: t_periodic() missing 2 required positional arguments: 'shift_x' and 'shift_y'

In [ ]:
t_1_max, t_2_max, energy, t_1, t_2 = t_periodic(dim_x, state, g)

In [ ]:
H, K, V = hamiltonian_dense(state, dim_x, g)

eig_val, eig_vec = np.linalg.eigh(H)

In [ ]:
energy_2d

In [ ]:
energy* 3

In [ ]:
np.min(eig_val) * 3

# Potential 2D

In [ ]:
state = 2

In [ ]:
def H_potential_2D(
    dim_x: int, dim_y: int, shift_x: int, shift_y: int, V, states: int=2
) -> np.ndarray:

    sites = dim_x * dim_y

    V_H = np.zeros((states**sites, states**sites), dtype=complex)

    for y in range(dim_y):
        for x in range(dim_x):

            tail_x = x + dim_x * y
            head_x = dim_x*((y + shift_y) % (dim_y)) + ((x + shift_x) % (dim_x))

            site_tail_x = np.minimum(tail_x, head_x)
            site_head_x = np.maximum(tail_x, head_x)

            n_lambda = states**(site_tail_x % (sites - 1))
            n_mu = states**(sites - 1 - site_head_x)
            n_nu = states**(np.abs(site_head_x - site_tail_x) - 1)

            if states**sites != n_lambda * n_mu * n_nu * states**2:
                raise ValueError("Wrong dimention")

            # Iterate through all elements of the Potential energy matrix operator.
            for q in range(states):
                for q_prime in range(states):
                    for p in range(states):
                        for p_prime in range(states):

                            # Calculate the flattened indices of the associated element.
                            row = p * states + q
                            col = p_prime * states + q_prime
                            val = V[row, col]

                            # Check if element is non zero.
                            if np.allclose(val, 0, atol=1e-25):
                                continue  # skip writing 0s

                            for Lambda in range(int(n_lambda)):
                                for mu in range(int(n_mu)):
                                    for nu in range(int(n_nu)):

                                        # Calculate the indices in the hamiltonian.
                                        i = (
                                            mu
                                            + q * n_mu
                                            + nu * states * n_mu
                                            + p * n_nu * n_mu * states
                                            + Lambda * n_nu * n_mu * states**2
                                        )
                                        j = (
                                            mu
                                            + q_prime * n_mu
                                            + nu * states * n_mu
                                            + p_prime * n_nu * n_mu * states
                                            + Lambda * n_nu * n_mu * states**2
                                        )

                                        # Assign a values to associated.
                                        V_H[i, j] += val
    return V_H

In [ ]:
def H_potential_2D_sparse(
    dim_x: int, dim_y: int, shift_x: int, shift_y: int, V, states: int=2
) -> np.ndarray:

    sites = dim_x * dim_y

    dim = state**sites
    rows = []
    cols = []
    data = []

    # Use COO for fast iteration
    V = V.tocoo()

    for y in range(dim_y):
        for x in range(dim_x):

            tail_x = x + dim_x * y
            head_x = dim_x*((y + shift_y) % (dim_y)) + ((x + shift_x) % (dim_x))

            site_tail_x = np.minimum(tail_x, head_x)
            site_head_x = np.maximum(tail_x, head_x)

            n_lambda = states**(site_tail_x % (sites - 1))
            n_mu = states**(sites - 1 - site_head_x)
            n_nu = states**(np.abs(site_head_x - site_tail_x) - 1)

            # Define place values for state based numerical system.
            stride_q = n_mu
            stride_nu = state * n_mu
            stride_p = n_nu * n_mu * state
            stride_lambda = n_nu * n_mu * state ** 2

            # Create a new list of row, column and data indecies.
            for row_2s, col_2s, val in zip(
                V.row, V.col, V.data
            ):  # Iterate only through non-zero elements of the
                # Unflatten the indices.
                p, q = divmod(row_2s, state)
                p_prime, q_prime = divmod(col_2s, state)

                for Lambda in range(n_lambda):
                    for mu in range(n_mu):
                        for nu in range(n_nu):
                            # Calculate the indices in the hamiltonian.
                            i = mu + q * stride_q + nu * stride_nu + p * stride_p + Lambda * stride_lambda
                            j = mu + q_prime * stride_q + nu * stride_nu + p_prime * stride_p + Lambda * stride_lambda

                            # Uppend them to a new rows, collumns and data arrays.
                            rows.append(i)
                            cols.append(j)
                            data.append(val)

    rows = np.asarray(rows, dtype=np.float64)
    cols = np.asarray(cols, dtype=np.float64)
    data = np.asarray(data, dtype=np.float64)

    # Return a new sparse array of the potential energy part of the mamiltonian.
    return sp.csr_matrix((data, (rows, cols)), shape=(dim, dim), dtype=np.float64)

In [ ]:
V = sp.csc_matrix(write_matrix_elements_new(state).real)

In [ ]:
dim_x = 3
dim_y = 3

In [ ]:
V_2D = H_potential_2D_sparse(dim_x, dim_y, 1, 1, V)

In [ ]:
V_old = H_potential_sparse(state, np.maximum(dim_x, dim_y), V, 1)

print("Done.")

eig_val_2D, _ = sp.linalg.eigsh(V_2D, k=1, which='SA', tol=1e-19, maxiter=2000)
eig_val_old, _ = sp.linalg.eigsh(V_old, k=1, which='SA', tol=1e-19, maxiter=2000)

print(eig_val_2D[0] - np.min(eig_val_old) * np.minimum(dim_x, dim_y))

# Potential in 3D

In [ ]:
def H_potential_3D(
    dim_x: int, dim_y: int, dim_z: int, shift_x: int, shift_y: int, shift_z: int, V, states: int=2
) -> np.ndarray:

    sites = dim_x * dim_y * dim_z

    V_H = np.zeros((states**sites, states**sites), dtype=complex)

    for z in range(dim_z):
        for y in range(dim_y):
            for x in range(dim_x):

                tail_x = x + dim_x * y + dim_y * dim_x * z
                head_x = dim_x*dim_y*((z + shift_z) % (dim_z)) + dim_x*((y + shift_y) % (dim_y)) + ((x + shift_x) % (dim_x))

                site_tail_x = np.minimum(tail_x, head_x)
                site_head_x = np.maximum(tail_x, head_x)

                n_lambda = states**(site_tail_x % (sites - 1))
                n_mu = states**(sites - 1 - site_head_x)
                n_nu = states**(np.abs(site_head_x - site_tail_x) - 1)

                if states**sites != n_lambda * n_mu * n_nu * states**2:
                    raise ValueError("Wrong dimention")

                # Iterate through all elements of the Potential energy matrix operator.
                for q in range(states):
                    for q_prime in range(states):
                        for p in range(states):
                            for p_prime in range(states):

                                # Calculate the flattened indices of the associated element.
                                row = p * states + q
                                col = p_prime * states + q_prime
                                val = V[row, col]

                                # Check if element is non zero.
                                if np.allclose(val, 0, atol=1e-25):
                                    continue  # skip writing 0s

                                for Lambda in range(int(n_lambda)):
                                    for mu in range(int(n_mu)):
                                        for nu in range(int(n_nu)):

                                            # Calculate the indices in the hamiltonian.
                                            i = (
                                                mu
                                                + q * n_mu
                                                + nu * states * n_mu
                                                + p * n_nu * n_mu * states
                                                + Lambda * n_nu * n_mu * states**2
                                            )
                                            j = (
                                                mu
                                                + q_prime * n_mu
                                                + nu * states * n_mu
                                                + p_prime * n_nu * n_mu * states
                                                + Lambda * n_nu * n_mu * states**2
                                            )

                                            # Assign a values to associated.
                                            V_H[i, j] += val
    return V_H

In [ ]:
def H_potential_3D_sparse(
    dim_x: int, dim_y: int, dim_z: int, shift_x: int, shift_y: int, shift_z: int, V, states: int=2
) -> np.ndarray:

    sites = dim_x * dim_y * dim_z

    dim = state**sites
    rows = []
    cols = []
    data = []

    # Use COO for fast iteration
    V = V.tocoo()

    for z in range(dim_z):
        for y in range(dim_y):
            for x in range(dim_x):

                tail_x = x + dim_x * y + dim_y * dim_x * z
                print(tail_x)
                head_x = dim_x*dim_y*((z + shift_z) % (dim_z)) + dim_x*((y + shift_y) % (dim_y)) + ((x + shift_x) % (dim_x))

                site_tail_x = np.minimum(tail_x, head_x)
                site_head_x = np.maximum(tail_x, head_x)

                n_lambda = states**(site_tail_x % (sites - 1))
                n_mu = states**(sites - 1 - site_head_x)
                n_nu = states**(np.abs(site_head_x - site_tail_x) - 1)

                # Define place values for state based numerical system.
                stride_q = n_mu
                stride_nu = state * n_mu
                stride_p = n_nu * n_mu * state
                stride_lambda = n_nu * n_mu * state ** 2

                # Create a new list of row, column and data indecies.
                for row_2s, col_2s, val in zip(
                    V.row, V.col, V.data
                ):  # Iterate only through non-zero elements of the
                    # Unflatten the indices.
                    p, q = divmod(row_2s, state)
                    p_prime, q_prime = divmod(col_2s, state)

                    for Lambda in range(n_lambda):
                        for mu in range(n_mu):
                            for nu in range(n_nu):
                                # Calculate the indices in the hamiltonian.
                                i = mu + q * stride_q + nu * stride_nu + p * stride_p + Lambda * stride_lambda
                                j = mu + q_prime * stride_q + nu * stride_nu + p_prime * stride_p + Lambda * stride_lambda

                                # Uppend them to a new rows, collumns and data arrays.
                                rows.append(i)
                                cols.append(j)
                                data.append(val)

    rows = np.asarray(rows, dtype=np.float64)
    cols = np.asarray(cols, dtype=np.float64)
    data = np.asarray(data, dtype=np.float64)

    # Return a new sparse array of the potential energy part of the mamiltonian.
    return sp.csr_matrix((data, (rows, cols)), shape=(dim, dim), dtype=np.float64)

In [ ]:
dim_x = 3
dim_y = 3
dim_z = 3

In [ ]:
state = 2

In [ ]:
V = sp.csc_matrix(write_matrix_elements_new(state).real)

In [ ]:
V_2D = H_potential_3D_sparse(dim_x, dim_y, dim_z, 1, 1, 1, V)

In [ ]:
V_old = H_potential_sparse(state, np.maximum(dim_x, dim_y), V, 1)

print("Done.")

eig_val_2D, _ = sp.linalg.eigsh(V_2D, k=1, which='SA', tol=1e-19, maxiter=2000)
eig_val_old, _ = sp.linalg.eigsh(V_old, k=1, which='SA', tol=1e-19, maxiter=2000)

print(eig_val_2D[0] - np.min(eig_val_old) * np.minimum(dim_x, dim_y))

In [ ]:
eig_val_2D[0]

In [ ]:
np.min(eig_val_old)*9